# MTGen — Mersenne Twister (MT19937) equidistribution

**Mersenne Twister** (Matsumoto & Nishimura, 1998) is the de-facto
standard PRNG for simulation work. The canonical MT19937 parameter
set has state size $k = wr - p = 32 \cdot 624 - 31 = 19937$ and
period $2^{19937} - 1$.

The default `tempMK2` tempering brings the gap profile down to SE = 0
(maximally equidistributed) at $L = 32$. This notebook builds the
bare generator first, runs the equidistribution test, then adds the
tempering chain and re-runs to show the improvement.

See [generators/MTGen.md](../../generators/MTGen.md) for the family
page and [theory/equidistribution-spec.md](../../theory/equidistribution-spec.md)
for the matricial method.


## Imports


In [ ]:
# stamp:auto-generated
from regpoly.core.generator import Generator
from regpoly.core.combination import Combination
from regpoly.core.transformation import Transformation
from regpoly.analyses.equidistribution_test import (
    EquidistributionTest,
    METHOD_MATRICIAL, METHOD_HARASE,
    METHOD_NOTPRIMITIVE, METHOD_SIMD_NOTPRIMITIVE,
)

INT_MAX = 2**31 - 1


## Construct the generator — _Matsumoto & Nishimura (1998)_


In [ ]:
# Canonical MT19937 parameters (Matsumoto & Nishimura, 1998 — Table 2).
gen = Generator.create("MTGen", L=32,
    w=32, r=624, m=397, p=31,
    a=0x9908B0DF)
print(gen.display())

# MT19937 tempering (Matsumoto-Kurita "tempMK2" form).
tempering = Transformation.create("tempMK2",
    w=32, eta=7, mu=15, u=11, l=18,
    b=0x9D2C5680, c=0xEFC60000)


## Wrap in a `Combination`


In [ ]:
# Wrap the generator + tempering chain in a single-component Combination.
comb = Combination(J=1, Lmax=gen.L)
comb.components[0].add_gen(gen)
comb.components[0].add_trans(tempering)
comb.reset()
print(f"k_g = {comb.k_g}, L = {comb.L}")


## Equidistribution test

MT19937's characteristic polynomial is primitive (full period), so the **Harase** primal-lattice method is the right choice — it is the fastest method for full-period generators.


In [ ]:
# Build the equidistribution test and run it. We cap `delta` at
# INT_MAX so nothing is rejected — we just want the score.
test = EquidistributionTest(
    L=gen.L,
    delta=[INT_MAX] * (gen.L + 1),
    mse=INT_MAX,
    method=METHOD_HARASE,
)
result = test.run(comb)

print(f"SE (Σ gaps)   = {result.se}")
print(f"verified      = {result.verified}")
print(f"first 10 gaps = {[result.ecart[i] for i in range(1, min(11, gen.L + 1))]}")


## Catalog entry

The published version of this parameter set lives in the REGPOLY catalog under `library_id = "mt19937"`. To load it programmatically without hard-coding parameters:

```python
from regpoly.library import Catalog
cat = Catalog('docs/library')
cat.load()
_, entry = cat.generator('mt19937')
# entry.components[0] carries the same params as constructed above
```
